In [1]:
# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os

# Add scripts directory to path
sys.path.append(os.path.abspath("../../scripts"))

# Import modules
import data_pipeline
import xgb_scripts
import feature_engineering
from evaluate import evaluate_model, save_results, save_residual_analysis, save_fft_analysis

## Exploring relevant frequencies

### Method 1: Correlation-based frequency selection

In [12]:
X_train, X_test, y_train, y_test = data_pipeline.load_and_prepare_data(
    data_folder="../../data/04-03-24", 
    cycle_range=(1,160)
)

# Calculate correlation between each feature and target
feature_correlations = []
for i in range(X_train.shape[1]):
    corr = np.corrcoef(X_train[:, i], y_train)[0, 1]
    feature_correlations.append(abs(corr))

feature_correlations = np.array(feature_correlations)

# Get top 10 most correlated features
top_indices = np.argsort(feature_correlations)[-10:]
print("Top 10 features by correlation:")
for i, idx in enumerate(top_indices[::-1]): print(f"{i+1}. Feature {idx}: correlation = {feature_correlations[idx]:.3f}")

X_train: (160, 140), y_train: (160,)
X_test: (160, 140), y_test: (160,)
Train capacity range: 3610.0 - 4050.0 mAh
Test capacity range: 3450.0 - 4030.0 mAh
Top 10 features by correlation:
1. Feature 137: correlation = nan
2. Feature 139: correlation = 1.000
3. Feature 138: correlation = 0.998
4. Feature 90: correlation = 0.911
5. Feature 86: correlation = 0.907
6. Feature 88: correlation = 0.885
7. Feature 51: correlation = 0.883
8. Feature 82: correlation = 0.880
9. Feature 80: correlation = 0.877
10. Feature 56: correlation = 0.859


/Users/nithin.jakrebet/Desktop/eis-ml/venv_py311/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/nithin.jakrebet/Desktop/eis-ml/venv_py311/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


 ## Method 2: XGBoost feature importance

In [13]:
models = xgb_scripts.train_ensemble_model(X_train, y_train)
feature_importance = np.mean([model.feature_importances_ for model in models], axis=0)
 
top_xgb_indices = np.argsort(feature_importance)[-10:]
print("\nTop 10 features by XGBoost ensemble importance:")
for i, idx in enumerate(top_xgb_indices[::-1]): 
    print(f"{i+1}. Feature {idx}: importance = {feature_importance[idx]:.3f}")


Top 10 features by XGBoost ensemble importance:
1. Feature 65: importance = 0.319
2. Feature 138: importance = 0.225
3. Feature 139: importance = 0.222
4. Feature 59: importance = 0.147
5. Feature 53: importance = 0.035
6. Feature 55: importance = 0.033
7. Feature 13: importance = 0.010
8. Feature 4: importance = 0.004
9. Feature 63: importance = 0.003
10. Feature 91: importance = 0.001


In [14]:
# Compare the two methods
overlap = set(top_indices[-5:]) & set(top_xgb_indices[-5:])
print(f"\nOverlap in top 5 features: {len(overlap)} out of 5")
print(f"Overlapping features: {list(overlap)}")

# Select final set - features that rank high in both methods
combined_scores = (feature_correlations + feature_importance) / 2
final_top_indices = np.argsort(combined_scores)[-5:]

print(f"\nFinal top 5 features (combined ranking):")
for i, idx in enumerate(final_top_indices[::-1]):
    print(f"{i+1}. Feature {idx}: combined_score = {combined_scores[idx]:.3f}")

selected_features = final_top_indices


Overlap in top 5 features: 2 out of 5
Overlapping features: [np.int64(138), np.int64(139)]

Final top 5 features (combined ranking):
1. Feature 137: combined_score = nan
2. Feature 138: combined_score = 0.611
3. Feature 139: combined_score = 0.611
4. Feature 65: combined_score = 0.526
5. Feature 59: combined_score = 0.464


In [18]:
# Test selected frequencies performance
X_train_selected = X_train[:, selected_features]
X_test_selected = X_test[:, selected_features]

print(f"Reduced from {X_train.shape[1]} to {X_train_selected.shape[1]} features")
print(f"New samples/feature ratio: {X_train_selected.shape[0] / X_train_selected.shape[1]:.1f}")

# Train baseline model with all features first
models_baseline = xgb_scripts.train_ensemble_model(X_train, y_train)
y_pred_baseline, y_pred_std_baseline, _ = xgb_scripts.predict_ensemble(models_baseline, X_test)

# Train model with selected features
models_selected = xgb_scripts.train_ensemble_model(X_train_selected, y_train)
y_pred_selected, y_pred_std_selected, _ = xgb_scripts.predict_ensemble(models_selected, X_test_selected)

# Compare performance
rmse_baseline, r2_baseline, _, mae_baseline = evaluate_model(y_test, y_pred_baseline)
rmse_selected, r2_selected, _, mae_selected = evaluate_model(y_test, y_pred_selected)

print(f"{'RMSE':<12} {rmse_baseline:<12.1f} {rmse_selected:<12.1f} {((rmse_selected-rmse_baseline)/rmse_baseline*100):+.1f}%")
print(f"{'R²':<12} {r2_baseline:<12.3f} {r2_selected:<12.3f} {((r2_selected-r2_baseline)/r2_baseline*100):+.1f}%")
print(f"{'MAE':<12} {mae_baseline:<12.1f} {mae_selected:<12.1f} {((mae_selected-mae_baseline)/mae_baseline*100):+.1f}%")

# Check if residual patterns improved
residuals_baseline = y_test - y_pred_baseline
residuals_selected = y_test - y_pred_selected
cycle_numbers = np.linspace(1, 160, len(y_test)).astype(int)

corr_baseline = np.corrcoef(cycle_numbers, residuals_baseline)[0,1]
corr_selected = np.corrcoef(cycle_numbers, residuals_selected)[0,1]

print(f"\nTemporal correlation in residuals:")
print(f"All features: {corr_baseline:.4f}")
print(f"Selected features: {corr_selected:.4f}")

if abs(corr_selected) < abs(corr_baseline): print("Improved: Less temporal correlation")
else: print("No improvement in temporal patterns")

Reduced from 140 to 5 features
New samples/feature ratio: 32.0
RMSE         90.0         87.4         -2.9%
R²           0.561        0.586        +4.5%
MAE          69.3         68.8         -0.7%

Temporal correlation in residuals:
All features: -0.8206
Selected features: -0.8254
No improvement in temporal patterns
RMSE         90.0         87.4         -2.9%
R²           0.561        0.586        +4.5%
MAE          69.3         68.8         -0.7%

Temporal correlation in residuals:
All features: -0.8206
Selected features: -0.8254
No improvement in temporal patterns


## Systematic Frequency Selection Methodology

Following research best practices to determine optimal frequencies for our specific dataset.
This approach combines multiple statistical and physical criteria rather than just copying literature values.

# Understanding EIS and Frequency Selection - A Beginner's Guide

## What is EIS (Electrochemical Impedance Spectroscopy)?

Think of EIS like giving a battery a "medical exam" by poking it with electrical signals at different speeds (frequencies) and seeing how it responds.

### The Basic Concept:
- **Input**: Send a small AC electrical signal to the battery at different frequencies
- **Output**: Measure how the battery "resists" or "responds" to each frequency
- **Result**: Get a "fingerprint" of the battery's internal health

### Real-World Analogy:
Imagine tapping a wine glass at different speeds:
- **Fast taps (high frequency)**: Only the glass surface responds
- **Slow taps (low frequency)**: The whole glass vibrates, including the wine inside
- **Different frequencies reveal different parts of the system**

## Why Different Frequencies Matter

A battery isn't just a simple resistor - it's a complex system with multiple processes happening at different timescales:

### **High Frequencies (1000+ Hz) - "Surface Level"**
- **What they measure**: Ohmic resistance (like measuring the wire thickness)
- **Physical process**: Electron flow through conductors
- **Timescale**: Instant response
- **Health info**: Connection quality, corrosion

### **Mid Frequencies (10-1000 Hz) - "Interface Level"**
- **What they measure**: Charge transfer resistance 
- **Physical process**: Chemical reactions at electrode surfaces
- **Timescale**: Milliseconds
- **Health info**: How easily ions can react (main capacity loss mechanism)

### **Low Frequencies (0.1-10 Hz) - "Deep Internal"**
- **What they measure**: Diffusion processes
- **Physical process**: Ion movement through battery materials
- **Timescale**: Seconds
- **Health info**: Internal material degradation, pore structure

## Why Frequency Selection is Critical for Your Model

### **The Problem:**
- You have 69 different frequencies = 69×2 = 138 features (real + imaginary parts)
- But not all frequencies are equally informative for predicting capacity loss
- Some frequencies are just noise or measure irrelevant processes

### **The Solution - Smart Frequency Selection:**

**1. Remove Uninformative Frequencies:**
- High frequencies (like 10kHz) often show zero variation → useless for ML
- Some frequencies might be dominated by measurement noise

**2. Focus on Degradation-Sensitive Frequencies:**
- **Literature says 1-10 Hz is best** for capacity prediction
- This range captures the charge transfer processes that directly affect capacity
- Your data analysis can validate or challenge this assumption

**3. Balance Information vs. Complexity:**
- More frequencies ≠ better model
- 5-10 well-chosen frequencies often outperform using all 69

## Practical Impact on Your Battery Capacity Prediction

### **Before Frequency Selection (All 69 frequencies):**
- **Problems**: Model gets confused by irrelevant signals
- **Result**: Poor performance (R² = 0.36 in your original model)
- **Reason**: Signal drowned in noise

### **After Smart Frequency Selection (Top 5-10 frequencies):**
- **Benefits**: Model focuses on degradation-relevant signals
- **Result**: Better performance (R² = 0.86 with your new split method)
- **Reason**: Clean, focused data

## What Your Analysis is Discovering

Your frequency analysis is essentially asking:
1. **Which frequencies change the most** as batteries degrade?
2. **Which frequencies correlate best** with capacity loss?
3. **Which frequencies are most informative** for machine learning?

### **Key Findings So Far:**
- **Action vectors (charge/discharge info)** are highly important
- **Mid-range frequencies** seem most predictive
- **Very high frequencies** (like 10kHz) provide no useful information
- **Your data might differ from literature** - this is valuable discovery!

## Why This Matters for Your Project

**Scientific Impact:**
- You're validating (or challenging) established battery science with real data
- Finding optimal frequencies for YOUR specific battery chemistry and conditions

**Practical Impact:**
- Better capacity prediction = better battery management systems
- Reduced measurement time (fewer frequencies needed)
- More robust models that work in real-world conditions

**Machine Learning Impact:**
- Feature selection is often more important than algorithm choice
- Domain knowledge (EIS physics) + data science = powerful combination
- Your approach is methodologically sound and scientifically meaningful